#### Run Uncertainty Estimation on causal patterns completed by unfinetuned GPT2 (zero-shot)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from geomechinterp.causal.base_functions import ALL_BINARY_GENERATORS
from geomechinterp.causal.pattern_generator import get_exhaustive_pattern_generators

In [3]:
print(ALL_BINARY_GENERATORS.keys()) # + their _prev modifications, that capture feature from the previous token in the sequence
# selected_features = ['position_parity', 'ab_prev', 'case_prev', '+-_prev', 'ab', 'case', '+-']
selected_features = ['position_parity', 'ab_prev', 'ab', 'case', '+-']

dict_keys(['position_parity', 'ab', 'case', '12', '+-', '><', ']['])


In [5]:
ALL_GENERATORS = get_exhaustive_pattern_generators(selected_features=selected_features,
                                                    verbose=True,
                                                    use_stochastic_features=False,
                                                    max_controls=2)

INFO:root:Generating all possible causal patterns for 5 features...
INFO:root:Found 32 unique causal patterns
100%|██████████| 32/32 [00:02<00:00, 11.74it/s]


In [6]:
from geomechinterp.causal.pattern_generator import generate_patterns_mp, pack_generator_and_pattern
ALL_GENERATORS = list(ALL_GENERATORS)

patterns = generate_patterns_mp(ALL_GENERATORS)
patterns[1] # Display first pattern

'B A B A B A B A B A'

In [7]:
# Create a dictionary mapping patterns to their generators
ALL_PATTERNS_AND_GENERATORS = {}
for pattern, generator in zip(patterns, ALL_GENERATORS):
    ALL_PATTERNS_AND_GENERATORS[pattern] = pack_generator_and_pattern(generator, pattern, selected_features)

In [ ]:
selected_features = ['position_parity', 'ab_prev', 'case_prev', '+-_prev', 'ab', 'case', '+-', "12_prev", "12"]

EXTENDED_GENERATORS_1 = get_exhaustive_pattern_generators(selected_features=selected_features,
                                                    verbose=True,
                                                    use_stochastic_features=True,
                                                    max_controls=2)

In [ ]:
selected_features = ['position_parity', 'ab_prev', 'case_prev', '+-_prev', 'ab', 'case', "><_prev", "><", "][_prev", "]["]

EXTENDED_GENERATORS_2 = get_exhaustive_pattern_generators(selected_features=selected_features,
                                                    verbose=True,
                                                    use_stochastic_features=True,
                                                    max_controls=1)

In [ ]:
from geomechinterp.causal.pattern_generator import generate_pattern
ALL_GENERATORS_EXTENDED = list(EXTENDED_GENERATORS_1) + list(EXTENDED_GENERATORS_2)

from multiprocessing import Pool

with Pool() as pool:
    patterns = pool.map(generate_pattern, ALL_GENERATORS_EXTENDED)

In [14]:
# Create a dictionary mapping patterns to their generators
ALL_PATTERNS_AND_GENERATORS_EXTENDED = {}
for pattern, generator in zip(patterns, ALL_GENERATORS_EXTENDED):
    ALL_PATTERNS_AND_GENERATORS_EXTENDED[pattern] = pack_generator_and_pattern(generator, pattern, selected_features)

In [16]:
ALL_PATTERNS_AND_GENERATORS_EXTENDED = ALL_PATTERNS_AND_GENERATORS_EXTENDED | ALL_PATTERNS_AND_GENERATORS

In [ ]:
len(ALL_PATTERNS_AND_GENERATORS), len(ALL_PATTERNS_AND_GENERATORS_EXTENDED)

In [22]:
# regenerate longer patterns
ALL_PATTERNS_AND_GENERATORS_FINAL = {}
for pattern, rec in ALL_PATTERNS_AND_GENERATORS_EXTENDED.items():
    extended_pat = generate_pattern(rec['generator'], pattern_length=30)
    rec['pattern'] = extended_pat
    ALL_PATTERNS_AND_GENERATORS_FINAL[extended_pat] = rec

In [24]:
import json
def save_patterns_and_generators(ALL_PATTERNS_AND_GENERATORS):
    # Convert generators to json format and save all patterns
    all_patterns_json = {
        pattern: {
            **{k: v for k, v in generator_dict.items() if k != "generator"},
            "generator": generator_dict["generator"].to_json()
        }
        for pattern, generator_dict in ALL_PATTERNS_AND_GENERATORS.items()
    }
    return all_patterns_json

json_data = save_patterns_and_generators(ALL_PATTERNS_AND_GENERATORS_FINAL)
json.dump(json_data, open('all_patterns_and_generators_final.json', 'w'), indent=2)

In [ ]:
list(json_data.keys())[:2]

In [29]:
data = json.load(open("./data/all_patterns_and_generators_final.json"))

In [61]:
test_data = {}
for i, (k, v) in enumerate(json_data.items()):
    if i > 1000:
        break
    test_data[k] = v

json.dump(test_data, open('all_patterns_and_generators_test.json', 'w'), indent=2)

In [ ]:
from geomechinterp.causal.mygpt import SymbolTokenizer, load_dataset, create_tokenize_function
from geomechinterp.causal.base_functions import ALL_SYMBOLS
tokenizer = SymbolTokenizer(ALL_SYMBOLS)
enc = tokenizer.encode('ab ab ab+ AB+- B- [] [!] >< !A! <<++ x')
tokenizer.decode(enc)


In [ ]:
dataset = load_dataset(json_data)

In [ ]:
tokenize_func = create_tokenize_function(tokenizer)
tokenized_dataset = dataset.map(tokenize_func, batched=True)

In [ ]:
tokenizer.decode(tokenized_dataset[0]['input_ids'])

In [ ]:
df = pd.DataFrame(ALL_PATTERNS_AND_GENERATORS, columns=['pattern', 'generator'])
df.groupby('pattern').size().sort_values(ascending=False).head(10)

In [46]:
with open('all_patterns_and_generators.json', 'w') as f:
    json.dump(all_patterns_json, f, indent=2)

In [ ]:
from geomechinterp.informat.entropy import estimate_entropy_char, estimate_entropy_token

In [ ]:
print('char entropy:', estimate_entropy_char('ab ab ab'))
print('token entropy:', estimate_entropy_token('ab ab ab'))

In [ ]:
deterministic_patterns_entropy = [estimate_entropy_token(pat) for pat in ALL_DETERMINISTIC_PATTERNS_LIST]
# sort patterns by entropy
[(x, round(ent,3)) for ent, x in sorted(zip(deterministic_patterns_entropy, ALL_DETERMINISTIC_PATTERNS_LIST))]


In [4]:
from functools import partial
from typing import List, Optional, Union

import json
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from matplotlib import pyplot as plt
import pandas as pd
import plotly.io as pio
import torch
from circuitsvis.attention import attention_heads
from fancy_einsum import einsum
from IPython.display import HTML, IFrame
from jaxtyping import Float
from tqdm import tqdm

import transformer_lens.utils as utils
from transformer_lens import ActivationCache, HookedTransformer
import json

# model = HookedTransformer.from_pretrained(
#     "gpt2-small",
#     center_unembed=True,
#     center_writing_weights=True,
#     fold_ln=True,
#     refactor_factored_attn_matrices=True,
# )

model = HookedTransformer.from_pretrained("gpt2", device="mps")

# Get the default device used
device: torch.device = utils.get_device()

INFO:datasets:PyTorch version 2.4.1 available.
INFO:datasets:TensorFlow version 2.13.0 available.
INFO:datasets:JAX version 0.4.37 available.
INFO:datasets:Apache Beam version 2.52.0 available.
/Users/solar/miniconda3/envs/pytorch/lib/python3.11/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loaded pretrained model gpt2 into HookedTransformer


In [3]:
# load gpt2-small from huggingface
from transformers import AutoModelForCausalLM, AutoTokenizer
model_hf = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
tokenizer_hf = AutoTokenizer.from_pretrained("gpt2")

In [ ]:
from transformer_lens.loading_from_pretrained import convert_hf_model_config
from geomechinterp.minterp.tflens.utils import load_gpt2_to_hooked_transformer 
from transformer_lens import HookedTransformerConfig

hook_config = HookedTransformerConfig(**convert_hf_model_config('gpt2'))

model_my_hoooked = load_gpt2_to_hooked_transformer(model_hf, hook_config=hook_config)


In [ ]:
from geomechinterp.minterp.tflens.utils import count_parameters
print("{:,} vs {:,}".format(count_parameters(model_hf), count_parameters(model)))
print('diff:', "{:,}".format((count_parameters(model_hf) - count_parameters(model)) / count_parameters(model_hf)))

In [44]:
# test that the two models are the same

input_ids = tokenizer_hf.encode("Hello, how are you?", return_tensors="pt").to(device)

input_ids = torch.tensor([[1, 2, 3, 3, 1, 2, 5, 6, 2, 10, 2, 3]]).to("mps")
# output_ids = model_hf.generate(input_ids, max_length=1)
# print(tokenizer_hf.decode(output_ids[0], skip_special_tokens=True))

output_hf = model_hf(input_ids)
output_tl = model(input_ids)
output_tl_my = model_my_hoooked(input_ids)


In [ ]:
model_hf.transformer.h[0].attn.c_attn.weight.shape
# > torch.Size([768, 2304])

In [ ]:
print(output_hf['logits'].shape, output_tl.shape)

torch.allclose(output_hf['logits'], output_tl)
for k in range(output_hf['logits'].shape[1]):
    # print(torch.cosine_similarity(output_hf['logits'][:, k, :], output_tl[:, k, :]))
    # compare after softmax
    # print(torch.cosine_similarity(torch.nn.functional.softmax(output_hf['logits'][:, k, :], dim=-1), torch.nn.functional.softmax(output_tl[:, k, :], dim=-1)))
    print(torch.cosine_similarity(torch.nn.functional.softmax(output_hf['logits'][:, k, :], dim=-1), torch.nn.functional.softmax(output_tl_my[:, k, :], dim=-1)))


In [ ]:
print(output_hf['logits'].shape, output_tl.shape)

torch.allclose(output_hf['logits'], output_tl)
for k in range(output_hf['logits'].shape[1]):
    # print(torch.cosine_similarity(output_hf['logits'][:, k, :], output_tl[:, k, :]))
    # compare after softmax
    print(torch.cosine_similarity(torch.nn.functional.softmax(output_hf['logits'][:, k, :], dim=-1), torch.nn.functional.softmax(output_tl[:, k, :], dim=-1)))


In [ ]:
output_hf['logits'].shape

In [ ]:
import seaborn as sns
sns.set_theme(style="whitegrid")

def get_uncertainty(logits):
    softmax_logits = torch.nn.functional.softmax(logits, dim=-1)
    entropy = -torch.sum(softmax_logits * torch.log(softmax_logits + 1e-8), dim=-1)
    entropy = entropy.cpu().detach().numpy()
    max_prob = softmax_logits.max(dim=-1).values.cpu().detach().numpy()
    return entropy, max_prob

def plot_uncertainty(logits, tokens:List[str]=None):
    entropy, max_prob = get_uncertainty(logits)
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))

    axs[0].plot(entropy, color='blue')
    axs[0].set_title('Entropy')
    axs[0].set_xlabel('Steps')
    axs[0].set_ylabel('Entropy Value')

    axs[1].plot(max_prob, color='orange')
    axs[1].set_title('Max Probability')
    axs[1].set_xlabel('Steps')
    axs[1].set_ylabel('Max Probability Value')
    
    # Annotate with tokens at each step
    if tokens is not None:
        for i, token in enumerate(tokens):
            axs[1].annotate(token, (i, max_prob[i]), textcoords="offset points", xytext=(0,10), ha='center')

    plt.tight_layout()
    plt.show()

In [ ]:
def run_model_on_pattern(model, pattern, exclude_first_k:int=0, device:str='mps'):
    token_ids = model.to_tokens(pattern).to(device)  # Convert tokens to token ids
    token_ids = token_ids[:, :512]  # Limit the sequence length if necessary

    # Forward pass through the model
    logits = model(token_ids, return_type="logits")
    # compute cross-entropy loss
    loss = torch.nn.functional.cross_entropy(logits[0, exclude_first_k:, :], token_ids[0, exclude_first_k:])
    return loss.item()

def run_model_on_pattern_and_plot(model, pattern, annotate_tokens:str = False, device:str='mps'):
    token_ids = model.to_tokens(pattern).to(device)  # Convert tokens to token ids
    token_ids = token_ids[:, :512]  # Limit the sequence length if necessary

    # Forward pass through the model
    logits = model(token_ids, return_type="logits")

    print(model.to_str_tokens(logits[0, :, :].argmax(dim=-1)))
    # print(logits.var(dim=-1))
    if annotate_tokens:
        plot_uncertainty(logits[0, :, :], tokens=model.to_str_tokens(logits[0, :, :].argmax(dim=-1)))
    else:
        plot_uncertainty(logits[0, :, :])
    return logits

In [ ]:
pattern = 'a a a a a a a a a a a a a a a a a a a a a a a a a'
loss = run_model_on_pattern(model, pattern)
print(loss)

In [ ]:
_ = run_model_on_pattern_and_plot(model, pattern)

In [ ]:
pattern = 'b b b b b b b b b b b b b b b b b b b b b b b b b'
_ = run_model_on_pattern_and_plot(model, pattern)

In [ ]:
pattern = 'a b a b a b a b a b a b a b a b a b a b a b a b a b'
_ = run_model_on_pattern_and_plot(model, pattern, annotate_tokens=True)

In [ ]:
pattern = '+ - + - + - + - + - + - + - + - + - + - + - + - + -'
_ = run_model_on_pattern_and_plot(model, pattern, annotate_tokens=True)

In [ ]:
patterns = ['a a a a a a a a a a a a a a a a a a a a a a a a a',
            'b b b b b b b b b b b b b b b b b b b b b b b b b',
            'a b a b a b a b a b a b a b a b a b a b a b a b a',
            'b a b a b a b a b a b a b a b a b a b a b a b a b',
            'A A A A A A A A A A A A A A A A A A A A A A A A A',
            'B B B B B B B B B B B B B B B B B B B B B B B B B',
            'B A B A B A B A B A B A B A B A B A B A B A B A B',
            'A B A B A B A B A B A B A B A B A B A B A B A B A',
            '+ - + - + - + - + - + - + - + - + - + - + - + - +',
            '- + - + - + - + - + - + - + - + - + - + - + - + -',
            '1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1',
            '2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2',
            '1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2',
            '2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1',
            'a1 a1 a1 a1 a1 a1 a1 a1 a1 a1 a1 a1 a1 a1 a1 a1', 
            'a2 a2 a2 a2 a2 a2 a2 a2 a2 a2 a2 a2 a2 a2 a2 a2',
            'a1 a2 a1 a2 a1 a2 a1 a2 a1 a2 a1 a2 a1 a2 a1 a2',
            'a2 a1 a2 a1 a2 a1 a2 a1 a2 a1 a2 a1 a2 a1 a2 a1',
            'a1 b1 a1 b1 a1 b1 a1 b1 a1 b1 a1 b1 a1 b1 a1 b1',
            'a2 b2 a2 b2 a2 b2 a2 b2 a2 b2 a2 b2 a2 b2 a2 b2',
            'a1 b2 a1 b2 a1 b2 a1 b2 a1 b2 a1 b2 a1 b2 a1 b2',
            'a2 b1 a2 b1 a2 b1 a2 b1 a2 b1 a2 b1 a2 b1 a2 b1']
losses = []
for pattern in patterns:
    loss = run_model_on_pattern(model, pattern, exclude_first_k=5)
    losses.append(loss)

pd.DataFrame({'pattern': [p[:20] for p in patterns], 'loss': losses})


In [ ]:
pattern = 'a2 b2 a2 b2 a2 b2 a2 b2 a2 b2 a2 b2 a2 b2 a2 b2'
_ = run_model_on_pattern_and_plot(model, pattern, annotate_tokens=True)

In [ ]:
pattern = 'a1 b2 a1 b2 a1 b2 a1 b2 a1 b2 a1 b2 a1 b2 a1 b2'
_ = run_model_on_pattern_and_plot(model, pattern, annotate_tokens=True)

In [ ]:
deterministic_patterns_entropy = [estimate_entropy_token(pat) for pat in ALL_DETERMINISTIC_PATTERNS_LIST]
# sort patterns by entropy
[(x, round(ent,3)) for ent, x in sorted(zip(deterministic_patterns_entropy, ALL_DETERMINISTIC_PATTERNS_LIST))][-10:]

In [ ]:
ALL_DETERMINISTIC_PATTERNS_AND_GENERATORS2 = {}
for pat, rec in ALL_DETERMINISTIC_PATTERNS_AND_GENERATORS.items():
    extended_pat = generate_pattern(rec['generator'], pattern_length=20)
    rec['pattern'] = extended_pat
    ALL_DETERMINISTIC_PATTERNS_AND_GENERATORS2[extended_pat] = rec

In [ ]:
pattern = '+ - + - + - + - + - + - + - + - + - + - + - + - + -'
_ = run_model_on_pattern_and_plot(model, pattern, annotate_tokens=True)

In [ ]:
ALL_DETERMINISTIC_PATTERNS_LIST = list(ALL_DETERMINISTIC_PATTERNS_AND_GENERATORS2.keys())

losses = []
for pattern in ALL_DETERMINISTIC_PATTERNS_LIST:
    loss = run_model_on_pattern(model, pattern[2:], exclude_first_k=5)
    losses.append(loss)

df = pd.DataFrame({'pattern': [p[:29] for p in ALL_DETERMINISTIC_PATTERNS_LIST], 'loss': losses})

In [ ]:
df.sort_values(by='loss', ascending=False)

In [ ]:
pattern = '+B -a -a +A +B -a -a +A +B -a -a +A +B -a -a +A +B -a -a +A'
print(ALL_DETERMINISTIC_PATTERNS_AND_GENERATORS2[pattern]['generator'])
_ = run_model_on_pattern_and_plot(model, pattern, annotate_tokens=True)

In [ ]:
# check that all symbols in patters match to a single token
symbols = ['a','b', '+', '-', 'A', 'B', '1', '2', '>', '<', '?', '!', '[', ']', '(', ')']
for symbol in symbols:
    if symbol not in model.tokenizer.get_vocab():
        print(f"Symbol {symbol} not found in tokens")

In [ ]:

# Function to tokenize patterns, run them through the model, and measure uncertainty
def run_model_on_pattern_and_plot(model, pattern, device):
    # Preprocess pattern
    tokens = pattern.replace("...", "").split()  # Split the pattern into individual tokens
    token_ids = model.to_tokens(tokens).to(device)  # Convert tokens to token ids
    token_ids = token_ids[:, :512]  # Limit the sequence length if necessary

    # Forward pass through the model
    logits, cache = model(token_ids, return_type="logits")
    
    # Calculate uncertainty (entropy over logits)
    softmax_logits = torch.nn.functional.softmax(logits, dim=-1)
    entropy = -torch.sum(softmax_logits * torch.log(softmax_logits + 1e-8), dim=-1)

    # Return uncertainty (logit entropy) for each step
    return entropy.cpu().detach().numpy()

# Run model on each pattern and register uncertainty
uncertainties = {}
for pattern in patterns:
    uncertainties[pattern] = run_model_on_pattern_and_plot(model, pattern, device)

# Display the uncertainties for each pattern
for pattern, uncertainty in uncertainties.items():
    print(f"Pattern: {pattern}\nUncertainty at each step:\n{uncertainty}\n")

In [ ]:
# check that all symbols in patters match to a single token
symbols = ['a','b', '+', '-', 'A', 'B', '1', '2', '>', '<', '?', '!', '[', ']', '(', ')']
for symbol in symbols:
    if symbol not in model.tokenizer.get_vocab():
        print(f"Symbol {symbol} not found in tokens")

In [ ]:

# Function to tokenize patterns, run them through the model, and measure uncertainty
def run_model_on_pattern_and_plot(model, pattern, device):
    # Preprocess pattern
    tokens = pattern.replace("...", "").split()  # Split the pattern into individual tokens
    token_ids = model.to_tokens(tokens).to(device)  # Convert tokens to token ids
    token_ids = token_ids[:, :512]  # Limit the sequence length if necessary

    # Forward pass through the model
    logits, cache = model(token_ids, return_type="logits")
    
    # Calculate uncertainty (entropy over logits)
    softmax_logits = torch.nn.functional.softmax(logits, dim=-1)
    entropy = -torch.sum(softmax_logits * torch.log(softmax_logits + 1e-8), dim=-1)

    # Return uncertainty (logit entropy) for each step
    return entropy.cpu().detach().numpy()

# Run model on each pattern and register uncertainty
uncertainties = {}
for pattern in patterns:
    uncertainties[pattern] = run_model_on_pattern_and_plot(model, pattern, device)

# Display the uncertainties for each pattern
for pattern, uncertainty in uncertainties.items():
    print(f"Pattern: {pattern}\nUncertainty at each step:\n{uncertainty}\n")